# 第12回 演習：PCAの読み解き（バイプロット）

## 前回からの接続

第11回では、400枚の顔画像を PCA にかけ、平均顔と固有顔という共通の辞書に分解しました。主成分がそのまま絵になったので、「この軸は明るさらしい」「この軸は向きらしい」と、目で見て意味を当てられました。

今日の相手は自動車の表データです。6つの変数を並べた表の主成分は、顔のようには絵になりません。作った軸に意味を読むには別の見せ方が要る——個体の座標（点）と変数の重み（矢印）を一枚に重ねた**バイプロット**が、それです。前回の最後で予告した「軸を言葉にする」作業を、ここでやります。

## 今日の分析目標

**車の個性を生む特徴を特定し、一枚の図で語りたい。**

第11回の演習では、顔画像をPCAにかけると、主成分が「固有顔」として目に見えました。今回は同じPCAを表データ（自動車）に向けます。表データの主成分は顔にはならないので、スコア（点）と負荷量（矢印）を重ねた**バイプロット**という一枚の図で読み解きます。この演習では、自動車データでバイプロットを自分の手で描き、車の個性と変数の関係を読み取ります。TODOに取り組みながら、最後の「目標に答えられたか」で振り返りましょう。


## 学習ゴール

この回を終えると、次のことができるようになります。

- スコア（点）と負荷量（矢印）が何個ずつあり、それぞれ何を表すのかを、区別して説明できる
- スコアと負荷量がどちらも同じ PC1–PC2 平面の座標であることから、なぜ一枚に重ねられるのかを言える
- スコアと負荷量を重ねたバイプロットを自分で描き、どの個体がどの変数に富むかを読み取れる
- 矢印どうしの角度から変数間の相関を読み、その読みが厳密なのは全次元のときだけだと、限定つきで言える
- 主成分に付けた名前が便利なあだ名にすぎないことを、符号の任意性と負荷量の中身を根拠に説明できる


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
import japanize_matplotlib

plt.rcParams['font.size'] = 11
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ
df = sns.load_dataset('mpg').dropna()
feat = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration']
X_s = StandardScaler().fit_transform(df[feat])
pca2 = PCA(n_components=2).fit(X_s)
print(f'PC1: {pca2.explained_variance_ratio_[0]:.1%}, PC2: {pca2.explained_variance_ratio_[1]:.1%}')
print(f'2軸で保つ情報: {pca2.explained_variance_ratio_[:2].sum():.1%}')

### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は色の濃いところを扱います。


In [ ]:
# 図：分析の地図（全14回のどこにいるか）
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
here = {3}
for i, (p, c, x) in enumerate(zip(phases, colors, [0.17, 0.5, 0.83]), start=1):
    on = i in here
    ax.text(x, 0.62, p, ha='center', va='center', fontsize=13 if on else 11,
            color='white', bbox=dict(boxstyle='round,pad=0.6', facecolor=c,
                                     alpha=0.95 if on else 0.25))
for x0, x1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(x1, 0.62), xytext=(x0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


## 1. 負荷量を確認する

各変数が第1・第2主成分にどう効くか（矢印の向き）を、数値で見ます。

### 深掘り：負荷量（矢印）とスコア（点）は別物——同じ主成分空間の裏と表

このあと描くバイプロットには、**矢印**と**点**の二種類が同居します。ここでつまずく人がとても多いので、まず「何がいくつあるのか」から丁寧にほどきます。前回（第11回）の固有顔で言えば、負荷量は**固有顔そのもの**、スコアは**各顔がその固有顔をどれだけ含むかの重み**でした。今日はそれを、表データの言葉で言い直します。

**数がそもそも違う**　この演習の変数は6個（`mpg, cylinders, displacement, horsepower, weight, acceleration`）、車は392台です。1節のコードセルで確認する**負荷量（loading）は変数ごとに1本**——だから矢印は6本。いっぽう**スコア（score）は車ごとに1点**——だから点は392個あります。「6本の矢印」と「392個の点」。この個数の違いが、そのまま両者の正体の違いです。矢印は**変数**の性質を、点は**個体（車）**の居場所を表します。

**式の骨子（1行で）**　標準化した車 $i$ のデータ $\tilde{x}_i$（6個の数字の並び）を、第 $k$ 主成分の向き $v_k$（負荷量を並べた単位ベクトル）に射影した値が、その車のスコアです。

$$
\text{スコア}\;s_{ik} \;=\; \tilde{x}_i \cdot v_k \;=\; \sum_{j} \tilde{x}_{ij}\, v_{k}[j]
$$

ここで $v_k[j]$ が「変数 $j$ が第 $k$ 主成分にどれだけ効くか」＝**負荷量**で、1節の負荷量の表の1マスです。点の座標は $(s_{i1}, s_{i2})$、矢印の座標は $(v_1[j], v_2[j])$。**同じ主成分の平面 $(\text{PC1}, \text{PC2})$ の上に、片や個体、片や変数を置いている**——だから一枚に重ねられる。これがバイプロットが成り立つ理由です（主成分の直交性やスペクトル分解を使って、点と矢印を同じ図に置いてよい条件まで厳密に詰めた議論は、バイプロットという図法そのものを提案した Gabriel の原論文（1971）に譲ります）。逆向きにたどれば $\tilde{x}_{ij} \approx \sum_k s_{ik}\, v_k[j]$、つまり「スコア×負荷量を足し戻すと元のデータに近づく」という、第11回の再構成と同じ関係がここにも生きています。

**この実データの数字で確かめる**　2成分での寄与率は **PC1 79.8%、PC2 12.1%、合わせて91.9%**（セットアップのセルの実行値）。6変数の情報の9割超が、たった2軸に乗っています。負荷量の表（次のセル）を読むと、PC1 は `displacement 0.44 / horsepower 0.43 / weight 0.43 / cylinders 0.43` が正、`mpg -0.40 / acceleration -0.29` が負。**排気量・馬力・重さ・気筒数が同じ向きに束**になり、燃費が逆向き——これが「大きさ・パワーの軸」の中身です。いっぽうスコアの例として、1台目の車（8気筒・排気量307・重さ3504lb の大型車）のスコアは PC1 が **約 +2.3**。大きく重い車なので、大きさ・パワーの正の側に置かれる、と読めます。製造国別の PC1 平均で見ても、アメリカ **+1.03**、ヨーロッパ **−1.58**、日本 **−1.82** と、大型のアメリカ車が正、小型の日本・欧州車が負にきれいに分かれます。

**点と矢印を一緒に読む——射影で車の値を当てる**　バイプロットの真価は、点と矢印を**別々にではなく重ねて**読むところにあります。ある車の点から、ある変数の矢印の向きへ**垂線を下ろした足の位置**（射影）が、その車のその変数の大小のめやすになります。矢印の指す先にある点ほど、その変数が大きい。たとえば右のほうに離れて座る大型のアメリカ車は、右向きの `weight`・`displacement` の矢印に深く射影されるので「重く排気量が大きい」、左向きの `mpg` にはマイナス側へ射影されるので「燃費が悪い」と一目で読めます。点だけでは「どの車が似ているか」しか分からず、矢印だけでは「どの変数が効くか」しか分からない。重ねてはじめて「**この車はこの特徴に富む**」が言えるのです。

**用語のひとこと（角度の話への布石）**　`sklearn` の `components_` が返すのは**長さ1に正規化した固有ベクトル**で、厳密には「負荷量」というより主成分の**係数（重み）**です。因子分析でいう本来の負荷量＝「変数と主成分の相関」は、これに $\sqrt{\text{固有値}}$ を掛けたもの。ふだんは向きだけ見れば十分なのでこの区別は気にしなくてよいのですが、**次の節の「矢印の角度＝相関」を厳密に言うときだけ**この $\sqrt{\text{固有値}}$ スケーリングが効いてきます。頭の隅に置いておいてください。


In [ ]:
load = pd.DataFrame(pca2.components_.T, index=feat, columns=['PC1', 'PC2'])
print(load.round(2).to_string())

### 図で見る：なぜ点と矢印を一枚に重ねられるのか

1節の負荷量の表の1行が、変数1本の座標 $(v_1[j],\ v_2[j])$ です。いっぽう車1台の座標は $(s_{i1},\ s_{i2})$。どちらも「PC1 の成分・PC2 の成分」という2つの数の組で、**同じ平面の上の位置**にほかなりません。3枚並べて確かめます。


In [ ]:
# 図：スコア（点）と負荷量（矢印）は、どちらも同じ PC1–PC2 平面の座標
S = pca2.transform(X_s)      # 392台 × 2 … 個体（車）の座標
L = pca2.components_.T       # 6変数 × 2 … 変数の座標
cars = [94, 52, 297]         # 大型アメリカ車・小型日本車・欧州車を1台ずつ
vnames = ['weight', 'mpg', 'acceleration']
k = 3.0                      # ③で矢印だけに掛ける共通の倍率（見やすさのため）
BLUE, RED = '#0066cc', '#e63946'

def frame(ax, title, xlim, ylim):
    ax.axhline(0, color='#bbb', lw=1); ax.axvline(0, color='#bbb', lw=1)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.set_title(title, fontsize=11)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.4))

# ① 個体を置く
frame(axes[0], '① 個体を置く：点の座標 $(s_{i1},\\ s_{i2})$', (-4.8, 6.6), (-2.6, 4.0))
for ci in cars:
    px, py = S[ci]
    axes[0].plot([px, px], [0, py], ls=':', color=BLUE, lw=1)
    axes[0].plot([0, px], [py, py], ls=':', color=BLUE, lw=1)
    axes[0].scatter([px], [py], s=70, color=BLUE, zorder=3)
    axes[0].annotate(f"{' '.join(df['name'].iloc[ci].split()[:2])}\n({px:.2f}, {py:.2f})",
                     (px, py), textcoords='offset points', xytext=(0, 10),
                     ha='center', fontsize=9, color=BLUE)

# ② 変数を置く（目盛りが①より1桁小さいことに注意）
frame(axes[1], '② 変数を置く：矢印の座標 $(v_1[j],\\ v_2[j])$\n※ 目盛りが①より1桁小さい',
      (-1.2, 1.65), (-0.65, 1.15))
for v in vnames:
    ax_, ay = L[feat.index(v)]
    axes[1].plot([ax_, ax_], [0, ay], ls=':', color=RED, lw=1)
    axes[1].plot([0, ax_], [ay, ay], ls=':', color=RED, lw=1)
    axes[1].arrow(0, 0, ax_, ay, color=RED, width=0.008, head_width=0.05,
                  length_includes_head=True, zorder=3)
    axes[1].annotate(f'{v}\n({ax_:.2f}, {ay:.2f})', (ax_, ay), textcoords='offset points',
                     xytext=(0, 8 if ay >= 0 else -24), ha='center', fontsize=9, color=RED)

# ③ 重ねる
frame(axes[2], f'③ 同じ PC1–PC2 平面だから重ねられる\n※ 矢印は{k:.0f}倍に伸ばして表示',
      (-5.4, 6.6), (-2.6, 4.0))
for ci in cars:
    px, py = S[ci]
    axes[2].scatter([px], [py], s=70, color=BLUE, zorder=3)
    axes[2].annotate(' '.join(df['name'].iloc[ci].split()[:2]), (px, py),
                     textcoords='offset points', xytext=(-9, -4), ha='right',
                     fontsize=9, color=BLUE)
for v in vnames:
    ax_, ay = L[feat.index(v)]
    axes[2].arrow(0, 0, k * ax_, k * ay, color=RED, width=0.03, head_width=0.18,
                  length_includes_head=True, zorder=3)
    axes[2].annotate(v, (k * ax_, k * ay), textcoords='offset points',
                     xytext=(7, 6 if ay >= 0 else -12), ha='left', fontsize=9, color=RED)
plt.tight_layout(); plt.show()


**この図の読み方**　青が個体（車）、赤が変数です。①は392台から3台だけを取り出し、点線でそれぞれの座標を軸へ落としたもの。buick electra は $(5.17,\ -0.48)$、toyota corolla は $(-2.71,\ 0.39)$、peugeot 504 は $(-2.04,\ 2.91)$——PC1 が大きいほど大型でパワーがある側、PC2 が大きいほど `acceleration` の値が大きい側です。②は同じやり方で変数を置いたもので、1節の負荷量の表から `weight` $(0.43,\ 0.29)$、`mpg` $(-0.40,\ -0.24)$、`acceleration` $(-0.29,\ 0.89)$ の3本を取り、原点から矢印を引いています。①と②は軸の名前も向きもまったく同じで、違うのは**そこに何を置いたか**だけ——だから③のように一枚に重ねられます。

ひとつだけ実務上の注意があります。②の目盛りをよく見てください。点の座標は数単位の広がりを持つ（この3台でも $-2.71$ から $5.17$ まで開いています）のに、負荷量は長さ1のベクトルの成分なので絶対値が1を超えません。**桁がひとつ違う**のです。そこで③では、矢印だけに共通の倍率（ここでは3倍）を掛けて伸ばしています。全部の矢印に同じ数を掛けるので、**向きも、矢印どうしの長さの比も変わりません**。TODO① のヒントにある `ld[i,0]*3` の `*3` が、この倍率です。③では、右上へ伸びた `weight` の矢印の側に大型の buick electra が、その反対側に小型の toyota corolla が、そして左上へ伸びた `acceleration` の矢印の側に peugeot 504 が座っています。


### TODO①：バイプロットを描く

スコア（点、製造国 `df['origin']` で色分け）と、負荷量（矢印）を重ねたバイプロットを描いてください。

In [ ]:
sc = pca2.transform(X_s)   # スコア（点）
ld = pca2.components_.T     # 負荷量（矢印）
# TODO: 点（scを origin で色分け）と、矢印（ldを原点から）を重ねて描いてください
# ヒント: 点は plt.scatter、矢印は plt.arrow(0, 0, ld[i,0]*3, ld[i,1]*3, ...) を feat の数だけ
# ヒント: 変数名は plt.text で矢印の先に添えると読みやすい
...

## 2. 矢印の関係を、数値で確かめる

バイプロットで同じ方向・反対方向を向いた変数の相関を、実際に計算します。

### TODO②：相関を計算して矢印と比べる

「重さ（weight）と排気量（displacement）」「燃費（mpg）と重さ（weight）」の相関をそれぞれ計算し、バイプロットの矢印の向き（同方向／反対方向）と一致するか確かめてください。

### 深掘り：なぜ矢印の角度が相関になるのか——厳密なのは「全次元」、2次元は近似

バイプロットの醍醐味は、**矢印どうしの角度**から変数間の相関まで読めることでした——同じ向きなら正の相関、反対向きなら負の相関、直角ならほぼ無相関。TODO② はこれを数値で裏づける作業です。ただしこの「角度＝相関」、**厳密に成り立つのは全次元のときだけ**で、私たちが描く2次元の図では**近似**にすぎません。ここを正しく線引きしておかないと、図を読みすぎて足をすくわれます。

**なぜ角度が相関になるのか（骨子）**　データは標準化済みなので、変数どうしの関係は**相関行列** $R$ にまとまっています。$R$ は対称行列なので固有値分解でき、$R = V \Lambda V^{\top}$。各変数に $\sqrt{\Lambda}$ を掛けた**変数ベクトル** $g_j = \sqrt{\Lambda}\,(V \text{の第}j\text{行})$ を割り当てると、二つの変数ベクトルの内積は

$$
g_i \cdot g_j \;=\; \sum_{k} \lambda_k\, v_k[i]\, v_k[j] \;=\; R_{ij} \;=\; \text{相関}(i, j),
\qquad \lVert g_j \rVert = \sqrt{R_{jj}} = 1
$$

長さがどれも1なので、内積そのものが $\cos(\text{角度})$。つまり**二本の変数ベクトルのなす角の余弦が、ぴたり相関に等しい**——これが「角度＝相関」の正体です。ただしこの等式は $k$ を**全6次元**まで足しているのがミソ。バイプロットは上位2次元しか描かないので、和が途中で打ち切られ、等号は $\approx$ に緩みます。

**近似が効くとき・外れるとき**　打ち切りの誤差は、**その変数が2次元の面にどれだけ乗っているか**で決まります。矢印が長い（面内によく収まる）変数どうしなら近似は良く、面から飛び出す変数では崩れます。下のセルで3ペアを、①実際の相関、②全次元での $\cos$、③図に描いた2次元の矢印の $\cos$ で並べると、次のようになります（実行値）。

| 変数ペア | 相関 | 全次元 $\cos$ | 描いた2次元 $\cos$ |
|---|---:|---:|---:|
| weight – displacement | **+0.93** | +0.93（厳密一致） | +0.94（良い近似） |
| mpg – weight | **−0.83** | −0.83（厳密一致） | −1.00（強調されすぎ） |
| mpg – acceleration | **+0.42** | +0.42（厳密一致） | **−0.23（符号が逆！）** |

全次元の $\cos$ は3ペアとも相関に**完全一致**します（前節で触れた $\sqrt{\text{固有値}}$ スケーリングの変数ベクトルを使えば厳密、というのがこれ）。問題は3列目です。`weight` と `displacement` はどちらも矢印が長くPC1に乗るので、2次元でも +0.94 とほぼ正確。ところが **`mpg` と `acceleration` は、図の上では角度が90°より開いて「やや反対〜直角」に見えるのに、本当の相関は +0.42 の正**。2次元の $\cos$ は −0.23 と、**符号まで逆に読めてしまう**のです。原因は、`acceleration` の矢印がほぼ PC2 一本（上向き）に張り付いていて、`mpg`（主に PC1）との**本当の弱い正の相関が、この2次元の角度には映らない**こと。`mpg`–`weight` が −1.00 と実際の −0.83 より極端に出るのも、同じ「2次元に潰した」ゆがみです。

**「長い矢印なら安心」ではない**　ここは誤解しやすい要所です。各変数が2次元の面にどれだけ乗っているか（矢印の長さの二乗＝共通性）を測ると、`acceleration` は **0.99** とむしろ最も長い部類、`mpg` も **0.81** と十分に長い。両方とも面にはよく乗っているのに、角度は当てになりませんでした。近似が崩れる原因は**二つ**あります——(1) 矢印が短い（面から外れて情報が落ちている）場合と、(2) 二本の矢印が**ほぼ直角＝真の相関が小さい**場合。後者は、わずかな射影のゆがみで直角の左右どちらにも転びうるため、**長い矢印どうしでも符号が反転しうる**。`mpg`–`acceleration` はまさに (2) で、真の相関 +0.42 が小さいがゆえに符号が脆いのです。「矢印が長ければ角度も信じてよい」は、半分しか正しくありません。

**だから、こう読む**　角度から相関を読んでよいのは、**長くて面内に乗った矢印どうし**（今回なら weight・displacement・horsepower・cylinders・mpg のような PC1 上の大物）に限ります。**短い矢印や、ほぼ直角に見える矢印は要注意**——`acceleration` のように、図の見かけと実際の相関が食い違うことがあります。だからこそ TODO② のように、気になったペアは**必ず数値で相関を確かめる**。「厳密なのは全次元、2次元は便利な近似」という一線が、バイプロットを読みすぎないための最初の歯止めです。


In [ ]:
# 深掘りの数値確認：矢印の角度 ↔ 変数の相関（全次元では厳密、描いた2次元は近似）
from numpy.linalg import norm
pcaF = PCA().fit(X_s)                 # 全6次元の主成分
Vf, ev = pcaF.components_.T, pcaF.explained_variance_
V2 = pca2.components_.T               # 図に描いた2次元の矢印（単位ベクトル）
cos = lambda a, b: float(a @ b / (norm(a) * norm(b)))
print(f'{"変数ペア":24s}{"相関":>8}{"全次元cos":>12}{"描いた2次元cos":>16}')
for a, b in [('weight', 'displacement'), ('mpg', 'weight'), ('mpg', 'acceleration')]:
    i, j = feat.index(a), feat.index(b)
    r  = np.corrcoef(X_s[:, i], X_s[:, j])[0, 1]
    cf = cos(Vf[i] * np.sqrt(ev), Vf[j] * np.sqrt(ev))  # √固有値でスケールした全次元ベクトル
    c2 = cos(V2[i], V2[j])                               # 図の矢印そのもの（2次元・単位）
    print(f'{a + "-" + b:24s}{r:+8.3f}{cf:+12.3f}{c2:+16.3f}')
# → 全次元cos は相関に厳密一致。2次元cos は mpg-acceleration で符号すら逆になる（読みすぎ注意）


In [ ]:
# TODO: weight と displacement、mpg と weight の相関を計算して表示してください
# ヒント: np.corrcoef(X_s[:, i], X_s[:, j])[0, 1]。i, j は feat.index('変数名') で得られます
...

## 3. 何個の主成分で製造国を当てられる？

上位いくつの主成分で、製造国がどれだけ当たるかを見ます。

### 深掘り：主成分を材料にする——PCR とスクリープロット、そして意味づけの罠

このあとのセルは、見た目より深いことをしています。**元の6変数ではなく、PCAで作った上位 $n$ 個の主成分スコアを材料にして、製造国を当てる**——これは回帰でいう**主成分回帰（PCR, principal component regression）**の分類版です。元の変数を直接使わず、いったんPCで座標を組み替えてから予測に回す、その発想を体験しています。

**なぜPCを材料にすると嬉しいのか**　第7回で見た双子（`temp` と `atemp`、相関0.99）を思い出してください。このデータでも `displacement`・`weight`・`cylinders`・`horsepower` はおおむね0.84〜0.95と互いにべったりで、そのまま回帰に入れると係数が暴れる**多重共線性**の温床です。ところが主成分は**互いに直交（無相関）**に作られるので、PCスコアを材料にすれば共線性が原理的に消え、しかも本数も減らせます。下のセルの結果は、**PC1だけで正解率0.678、6個すべてでも0.722**。たった1軸（大きさ・パワーの軸）で製造国の見分けの大半が付き、軸を足しても伸びはわずか。「国ごとの車づくりの違い」は、ほぼこの1本に凝縮されているわけです。

**発展：何本残すか——スクリープロットで決める**　主成分を何本使うかの目安が**スクリープロット**（がれ場の図）です。各主成分の寄与率を大きい順に並べると、このデータでは **0.80, 0.12, 0.04, 0.02, 0.01, 0.01**。PC1 で崖のように落ち、PC2 のあたりで**肘（elbow）**を折り、その先は小石（scree）のように平ら——この「急落→平坦」の肘までを残す、という読み方です。累積では2本で92%に届くので、可視化には2軸で十分。PCR のように予測が目的なら、肘を目安にしつつ最後は交差検証（第5回）で本数を選ぶのが実務です（PCR を回帰手法の一つとして定式化し、成分数の選び方まで含めて扱った議論は、次元削減を使った回帰を章立てで解説した統計的学習の入門書——James ら『An Introduction to Statistical Learning』の「次元削減法」の節に譲ります）。

**PCR の落とし穴もひとつ**　主成分は**目的変数を見ずに**、分散の大きさだけで選ばれます。だから「分散は小さいが予測には効く方向」があると、上位成分だけ残す PCR はそれを取りこぼす恐れがあります。今回は幸い、最大分散の大きさ・パワー軸（PC1）が製造国とも強く結びついていたので PC1 がそのまま効きました。でも**分散の大きさと予測への効きは別物**——いつも上位成分が効くとは限りません。PCR を使うときは、成分を分散順に足すだけでなく、交差検証で「本当に効く成分か」を見張るのが安全です。

**つまずきどころ：主成分に「意味」を読み込みすぎない**　私たちは PC1 に「大きさ・パワー」、PC2 に「加速の軸」と名前を付けました。解釈の助けになる一方で、これは**行き過ぎると危険**な操作です。三つ、釘を刺しておきます。

- **符号に絶対的な意味はない**　固有値方程式は $v$ でも $-v$ でも成り立つので、主成分の向きの符号はライブラリの都合で決まっているだけ。実行環境が変われば図が左右反転し、「正が大型」が「負が大型」に化けます。読むべきは**軸（向き）であって符号ではない**。
- **PC1 は5変数の混ぜ物であって「大きさ」という実体ではない**　PC1 の負荷量は `0.44, 0.43, 0.43, 0.43, −0.40, −0.29` と6変数の重み付き和。「大きさ・パワー」はその混合物に人間が付けた**あだ名**で、世界のどこかに「大きさ変数」が実在するわけではありません。
- **きれいな軸ほど疑う**　PC2 は `acceleration` の負荷量が 0.89 とほぼ単独で、いかにも「加速の軸」に見えます。でもこれは、加速が大きさクラスタとの相関が弱いために、PCA が**余った軸に自動で置いただけ**。データの相関構造の副産物であって、「加速という法則」を発見したわけではありません。

要は、**主成分の名前は説明のための便利なラベルであって、実在する対象ではない**。名付けは歓迎ですが、主張はいつも数字（負荷量・相関・寄与率）に錨を下ろす——TODO② で相関を計算して裏を取るのは、まさにこの錨のためです。控えめな解釈、これが読み解きの鉄則です。


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
y = df['origin'].values
for n in [1, 2, 6]:
    acc = cross_val_score(Pipeline([('pca', PCA(n_components=n)),
                                    ('m', LogisticRegression(max_iter=5000))]), X_s, y, cv=5).mean()
    print(f'PC {n}個: 製造国の正解率 {acc:.3f}')

## 目標に答えられたか

- 今日の目標は「車の個性を生む特徴を特定し、一枚の図で語りたい」でした
- TODO①のバイプロットで、アメリカ車と日本・ヨーロッパ車は、どちら側に分かれましたか？
- 右を向く矢印（大きさ・パワーの変数群）は、どの製造国の車の方向を指していましたか？
- TODO②で、矢印の向き（同方向／反対方向）と、相関の符号は一致しましたか？
- PC1（第1主成分）に、あなたなら何という「名前」をつけますか？

## 今日の要点

- 負荷量は変数ごとに1本（6本）、スコアは個体ごとに1点（392点）。数が違うのは、片方が変数の性質を、もう片方が個体の居場所を表しているから
- 点の座標 $(s_{i1},\ s_{i2})$ も矢印の座標 $(v_1[j],\ v_2[j])$ も、同じ PC1–PC2 平面の2数の組である。バイプロットが一枚に重ねられる根拠は、ここにしかない
- 点と矢印は別々にではなく重ねて読む。ある点から矢印の向きへ下ろした射影の深さが、その個体のその変数の大小のめやすになる
- 矢印の角度が相関に一致するのは全次元での話。描いた2次元は近似で、`mpg`–`acceleration` のように符号まで逆に見えることがある。気になるペアは必ず数値で確かめる
- 主成分は互いに無相関だから、材料に使えば多重共線性が原理的に消える。これが PCR の利点で、このデータでは PC1 だけで製造国の正解率 0.678 に届いた
- ただし主成分は目的変数を見ずに、分散の大きさだけで選ばれる。分散は小さいが予測に効く向きがあると取りこぼす——PLS との差は、そこから生まれる
- 主成分に付けた名前は、説明のための便利なあだ名であって実在する対象ではない。主張はいつも負荷量・相関・寄与率という数字に錨を下ろす


## 次回へ

たくさんの変数を少数の軸に束ね、その軸に意味を読む——第11回と今日で、次元削減という道具を一周しました。6つの変数の情報が2本の軸で 91.9% まで説明でき、その2本に「大きさ・パワー」「加速」という名前まで付けられました。**列（変数）の方向にデータを要約する道具**、と言い換えてもよいでしょう。

次回は、要約する向きが変わります。列ではなく**行**——一つひとつの個体を、似た者どうしの群れに分ける。これがクラスタリングです。データも自動車から142か国の統計に移ります。正解ラベルがないのは今日と同じですが、悩みどころは新しくなります。当てる相手がないのに「うまく分かれた」とどうやって判定するのか。そして、群れはいくつに分ければよいのか。教師なし学習でいちばん厄介な問いが、そこで出てきます。


## 課題（提出）

**提出するもの**: 応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

冒頭のセルでは、標準化した `X_s` で PCA をしました。今度は標準化を**しません**。
元の値 `df[feat]` をそのまま `PCA(n_components=2)` に学習させ、PC1・PC2 の寄与率と、1節と同じ負荷量の表（変数名つき）を表示してください。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

冒頭のセルの `PCA(n_components=2).fit(X_s)` の `X_s` を `df[feat]` に変えるだけです。寄与率は `.explained_variance_ratio_`、負荷量の表は1節のセルと同じ書き方で `.components_.T` から作れます。

</details>


### 応用②（判断）

応用①で学習した、標準化なしの第1主成分（PC1）の寄与率は何%ですか。**%で小数第1位まで**答えてください（例: 45.6）。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

`explained_variance_ratio_[0]` を 100 倍して `round(値, 1)` です。冒頭のセルは `:.1%` の書式で同じ種類の値（寄与率）を同じ書式で表示しています。

</details>


### 応用③（解釈）

標準化をやめると、第1主成分の中身は冒頭のセルの「大きさ・パワーの軸」から大きく変わりました。
なぜ PCA の前に標準化が要るのか。応用①の負荷量の絶対値が最も大きい変数が何か、その変数の単位（たとえば馬力なら hp）と値の大きさに触れて、「標準化は面倒だから省きたい」と言う同級生に向けて3行で説明してください。


（ここに3行程度で書く）


<details><summary>詰まったら</summary>

`df[feat].std()` で6変数の標準偏差を並べると、どの変数がどれだけ大きく動くかが見えます。PCA は「ばらつきが最も大きい向き」を探す手法だったことを思い出してください。

</details>


## 発展（任意）

### 圧縮してから予測する：主成分回帰（PCR）と PLS

3節では、主成分スコアを材料に製造国を当てました。同じ発想は回帰にも使えます。目的変数を `mpg`（燃費）、説明変数を残りの5変数にすると、`cylinders`・`displacement`・`horsepower`・`weight` は互いの相関が 0.84〜0.95 と非常に強く、第7回の双子（`temp` と `atemp`）と同じ多重共線性の問題を抱えています。

第7回では正則化でこれをしのぎました。もう一つの手が、**先に主成分に圧縮してから回帰する**ことです。主成分どうしは無相関なので、共線性が原理的に消えます。これが**主成分回帰（PCR, principal component regression）**です。

ただし PCR の軸は目的変数を見ずに決まります（3節の深掘りで触れた落とし穴）。そこで、**目的変数との共分散が大きい向き**を軸にする **PLS（partial least squares, 部分的最小二乗）** という手法があります。化学や計測の分野では、説明変数が数百本あるスペクトルデータの標準手法です。

以下では、素の重回帰・PCR・PLS の3つを、同じ5分割交差検証の R² で比べます。


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold

pred = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration']
Xp, yp = df[pred].values, df['mpg'].values
cv = KFold(n_splits=5, shuffle=True, random_state=0)
models = {
    '重回帰（5変数そのまま）': Pipeline([('sc', StandardScaler()), ('m', LinearRegression())]),
    'PCR（主成分2個→回帰）': Pipeline([('sc', StandardScaler()), ('pca', PCA(n_components=2)),
                                       ('m', LinearRegression())]),
    'PLS（成分2個）': Pipeline([('sc', StandardScaler()), ('m', PLSRegression(n_components=2))]),
}
for name, m in models.items():
    r2 = cross_val_score(m, Xp, yp, cv=cv, scoring='r2')
    print(f'{name:16s} CV R² = {r2.mean():.3f} ± {r2.std():.3f}')


**読み方**　3つの R² はいずれも 0.68〜0.69 で、ほぼ横並びです。5本の変数を2本の軸に圧縮しても、燃費の当たり具合はほとんど落ちていません。強く相関した4変数が持つ情報は、実質1〜2本の軸に収まっているからです。

同じ2成分でも、PLS（0.688）は PCR（0.683）よりわずかに上です（差は fold 間のばらつき（±0.03）の中に収まる程度です）。PCR が「分散の大きい向き」を機械的に選ぶのに対し、PLS は目的変数 `mpg` を見ながら軸を作るので、少ない成分でも予測に効く向きを拾えます。

差が小さいのは、このデータでは最大分散の軸（大きさ・パワー）がそのまま燃費にも効いているからです。分散は小さいが予測には効く向きが隠れているデータほど、PLS の利点が大きくなります。

成分を何個にするかは、3節の深掘りで見たスクリープロットを目安にしつつ、最後は交差検証で決めます。PCR と PLS が理論的に何を違えているのかは、この節の最後の深掘り「PCR と PLS の違いは、軸を作るときに目的変数を見るかどうか」で骨子を追います（重みベクトルの求め方や、成分を一つ作るごとに残差へ移る反復まで含めた定式化は、PLS を回帰の枠組みで整理した計量化学（ケモメトリクス）の総説論文に譲ります）。

試すなら、このデータでは、`n_components` を 1 に減らしてみてください。PCR は 0.656、PLS は 0.665 と、成分が少ないほど「目的変数を見て軸を作る」PLS の差が開きます。


### 深掘り：PCR と PLS の違いは、軸を作るときに目的変数を見るかどうか

**軸の作り方だけが違う**　PCR と PLS は、「5本の変数を少数の軸に圧縮してから回帰する」という骨格は同じです。違うのは軸の作り方ひとつ。PCR の軸は PCA が作るので、**説明変数の分散が大きい向き**から順に決まります。このとき目的変数 `mpg` は一度も参照されません——軸を作る段階と、その軸で回帰する段階が、完全に切り離されているのです。いっぽう PLS は、**目的変数との共分散が大きい向き**を軸にします。共分散は「説明変数側の散らばりの大きさ × 目的変数との連動の強さ」なので、PLS の軸は分散と予測への効きの折衷になります。よく散らばっていて、しかも `mpg` とよく連動する向きを、最初から狙って作りにいくわけです。

**だから、差が出るのはこういうとき**　両者に差が出るのは、**分散は小さいが予測には効く方向**がデータにあるときです。その向きは分散の順では後回しにされるので、上位数本しか使わない PCR は取りこぼします。PLS は目的変数を見ているので、分散が小さくても `mpg` と連動していれば上位に拾い上げる。逆に、いちばん散らばっている向きがそのまま目的変数にも効いているデータでは、両者はほとんど同じ軸を作るので、差はほとんど出ません。

**この発展の交差検証の実行値で確かめる**　このデータは後者に近い例です。成分2個では PCR 0.683、PLS 0.688 で、差 0.005 は fold 間のばらつき（±0.03）に埋もれています。差が見えてくるのは、軸を1本に絞って「1本目に何を選ぶか」だけで勝負させたとき——PCR 0.656、PLS 0.665 と、差は 0.009 に広がります。PCR の1本目は「大きさ・パワーの軸」、つまり燃費を見ずに、説明変数がいちばん散らばる向きとして選ばれたもの。PLS の1本目は、そこから燃費との連動のぶんだけ向きをずらした軸です。本数を増やせば PCR も後の成分で必要な向きを回収できるので差は縮む——**目的変数を見て軸を作る利点は、成分を絞るほど効いてくる**、と読めます。
